In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab.ipynb")

# Lab 9 – Models and Pipelines 🔁

## DSC 80, Winter 2026

### Due Date: Monday, March 9th at 11:59pm

## Instructions

Welcome to the ninth and final DSC 80 lab this quarter!

Much like in DSC 10, this Jupyter Notebook contains the statements of the problems and provides code and Markdown cells to display your answers to the problems. Unlike DSC 10, the notebook is *only* for displaying a readable version of your final answers. The coding will be done in an accompanying `lab.py` file that is imported into the current notebook, and **you will only submit that `lab.py` file**, not this notebook!

Some additional guidelines:
- **Unlike in DSC 10, labs will have both public tests and hidden tests.** The bulk of your grade will come from your scores on hidden tests, which you will only see on Gradescope after the assignment deadline.
- **Do not change the function names in the `lab.py` file!** The functions in the `lab.py` file are how your assignment is graded, and they are graded by their name. If you changed something you weren't supposed to, you can find the original code in the [course GitHub repository](https://github.com/dsc-courses/dsc80-2026-wi).
- Notebooks are nice for testing and experimenting with different implementations before designing your function in your `lab.py` file. You can write code here, but make sure that all of your real work is in the `lab.py` file, since that's all you're submitting.
- You are encouraged to write your own additional helper functions to solve the lab, as long as they also end up in `lab.py`.

**To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.** More details on its usage are given at the bottom of this notebook.

**Importing code from `lab.py`**:

* Below, we import the `.py` file that's contained in the same directory as this notebook.
* We use the `autoreload` notebook extension to make changes to our `lab.py` file immediately available in our notebook. Without this extension, we would need to restart the notebook kernel to see any changes to `lab.py` in the notebook.
    - `autoreload` is necessary because, upon import, `lab.py` is compiled to bytecode (in the directory `__pycache__`). Subsequent imports of `lab` merely import the existing compiled python.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from lab import *

In [3]:
import pandas as pd
import numpy as np
np.set_printoptions(legacy='1.21')
from pathlib import Path
import plotly.express as px

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from pipeline_testing_util import get_transformers

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

import warnings
warnings.simplefilter('ignore')

# DSC 80 preferred styles
pio.templates["dsc80"] = go.layout.Template(
    layout=dict(
        margin=dict(l=30, r=30, t=30, b=30),
        autosize=True,
        width=800,
        height=500,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True),
        title=dict(x=0.5, xanchor="center"),
    )
)
pio.templates.default = "simple_white+dsc80"

## Part 1: `sklearn` Pipelines 🧠

The file `data/toy.csv` contains an example dataset that consists of 4 columns:

- `'group'`: a categorical column with 3 categories
- `'c1'`: a numeric attribute
- `'c2'`: a numeric attribute
- `'y'`: the target variable (that you want to predict) 

In the following questions, you will build `Pipeline`s that combine feature engineering with a linear regression model.

In [5]:
fp = Path('data') / 'toy.csv'
data = pd.read_csv(fp)
data.head()

### Question 1

First, you will train a regression model using only a *log-scaled* `'c2'` variable. Create a `Pipeline` that:
1. log-scales `'c2'`, then
2. predicts `'y'` using a linear regression model (using your transformed `'c2'`).

That is, complete the implementation of the function `simple_pipeline`, which takes in a DataFrame like `data` and returns a **tuple** consisting of 
- An already-fit `Pipeline`, and
- An array containing the predictions your model makes on `data` (after being trained on `data`).

***Note***: By "log", we're referring to the natural logarithm. In order to log-scale `'c2'`, you'll need to use a `FunctionTransformer`.

In [7]:
# don't change this cell, but do run it -- it is needed for the tests to work
q1_fp = Path('data') / 'toy.csv'
q1_data = pd.read_csv(q1_fp)
q1_pl, q1_preds = simple_pipeline(q1_data)

In [ ]:
grader.check("q1")

### Question 2

Now, you will engineer features from the other columns and use them to train a regression model.  Create a `Pipeline` that:
1. uses `'c1'` as is,
1. log-scales `'c2'`,
1. one-hot encodes `'group'`, and
1. predicts `'y'` using a linear regression model built on the three variables above. (Note that your model will have more than three "features", because one-hot encoding `'group'` will create multiple columns. Don't drop any of them.)

That is, complete the implementation of the function `multi_type_pipeline`, which takes in a DataFrame like `data` and returns a **tuple** consisting of
- An already-fit `Pipeline`, and
- An array containing the predictions your model makes on `data` (after being trained on `data`).

***Hint***: Use `ColumnTransformer`, as we did in [Lecture 15](https://dsc80.com/resources/lectures/lec15/lec15.html#Planning-our-first-ColumnTransformer).

In [16]:
# don't change this cell, but do run it -- it is needed for the tests to work
q2_fp = Path('data') / 'toy.csv'
q2_data = pd.read_csv(q2_fp)
q2_pl, q2_preds = multi_type_pipeline(q2_data)

In [ ]:
grader.check("q2")

### Question 3

It seems like `'c1'` and `'c2'` have strong associations with the values of `'group'` (to see this, run the cell below). This suggests that group-wise scaling might make good features. 


Now, we want to standardize (i.e. z-score) both `'c1'` and `'c2'` **within each `'group'`** (`'A'`, `'B'`, and `'C'`). Unfortunately, there is no built-in transformer in `sklearn` that performs group-wise standardization, so **you will need to create your own transformer!**

Your job is to complete the implementation of the `StdScalerByGroup` transformer class, meaning that you need to implement the `fit` and `transform` methods, along with the constructor (`__init__`).
- The `StdScalerByGroup` transformer works on an input array/DataFrame `X` whose first column contains groups, and whose remaining columns are quantitative and need to be standardized (within each group).
- The `fit` method should determine the mean and standard deviation of each quantitative column within each group in the input data `X` and save them in the instance variable `grps_`. (For instance, one of the quantities you may calculate here is the standard deviation of `'c1'`, but only for the rows whose `'group'` is `'B'`.)
- The `transform` method should take in an input array/DataFrame `X`, standardize each quantitative column separately using the means and standard deviations stored in `grps_`, and return a DataFrame containing the transformed quantitative columns.


For example, if you `fit` and `transform` a `StdScalerByGroup` transformer on `data` (without the `'y'` column), you should get back a DataFrame with two columns, `'c1'` and `'c2'`.


***Notes:***
1. You may decide on whatever structure you'd like for the `grps_` variable. This question will be graded on the correctness of the output of your transformer. (Check the correctness of your work by checking the output by-hand!)    
2. You may use loops, but they are not necessary. Our solution uses `.apply`.
3. In the public tests, the column with group labels is named `'g'` instead of `'group'`. Remember, the first column will **always** contain the groups, even if the first column's name is something other than `'group'`.
4. Don't worry about cases where the standard deviation is 0.

In [29]:
# The scatter plot referenced at the start of Question 3
# This is not needed to answer the question, but motivates why we are standardizing
px.scatter(data, x='c1', y='y', color='group')

In [30]:
# don't change this cell, but do run it -- it is needed for the tests to work
# test fit 
q3_test_fit_cols = {'g': ['A', 'A', 'B', 'B'], 'c1': [1, 2, 2, 2], 'c2': [3, 1, 2, 0]}
q3_test_fit_X = pd.DataFrame(q3_test_fit_cols)
q3_test_fit_std = StdScalerByGroup().fit(q3_test_fit_X)

# test transform
q3_test_transform_cols = {'g': ['A', 'A', 'B', 'B'], 'c1': [1, 2, 3, 4], 'c2': [1, 2, 3, 4]}
q3_test_transform_X = pd.DataFrame(q3_test_transform_cols)
q3_test_transform_std = StdScalerByGroup().fit(q3_test_transform_X)
q3_test_transform_out = q3_test_transform_std.transform(q3_test_transform_X)

In [31]:
# don't change this cell, but do run it -- it is needed for the tests to work
q3_fit_data = pd.read_csv(Path('data') / 'toy.csv')

N = 2*10**3
a = np.random.choice(['A', 'B'], size=(N,1)).astype('object')
b = np.random.multivariate_normal([1, 2], [[1, 0],[0, 100]], size=N)
arr = np.hstack([a, b])
q3_transform_data = pd.DataFrame(arr)
q3_transform_data[1] = q3_transform_data[1].astype(float)
q3_transform_data[2] = q3_transform_data[2].astype(float)

In [ ]:
grader.check("q3")

### Question 4

`Pipeline`s are supposed to help you easily try different model configurations. Complete the implementation of the function `eval_toy_model`, which returns a **hard-coded list of tuples** consisting of the (RMSE, $R^2$) of three different modeling `Pipeline`s, fit and evaluated on the entire input dataset `data`. The three different `Pipeline`s are:
1. The `Pipeline` in Question 1.
1. The `Pipeline` in Question 2.
1. A `Pipeline` consisting of a linear regression model fit on features generated by applying `StdScalerByGroup` to `'c1'`, log-scaling `'c2'`, and applying `OneHotEncoder` to `'group'`.

Your function should return a list of length 3. Each element should be a 2-tuple of hard-coded numbers. Do not round.

In [ ]:
grader.check("q4")

## Part 2: Hyperparameter selection 🎛

### Question 5

In this question, you will train two new types of prediction models – **decision tree regressors and k-Nearest Neighbor regressors** – and explore how different hyperparameter choices affect the model's ability to generalize to new data. You'll use Galton's child heights dataset, which you've seen before in this class and in DSC 10.

#### `tree_reg_perf` 🌲

A decision tree regressor is trained similarly to a decision tree classifier: the splits of the tree are created by minimizing the variance of the (training data) response values in the leaves given by making the split in question. A decision tree regressor predicts the response value of a (new) observation based on the **average response value** of the training observations lying in the same leaf node. 

One **hyperparameter** of a decision tree regressor that affects model complexity is the **depth** of the tree. Larger depths correspond to more complicated decision trees. We will explore this parameter in this question.

Complete the implementation of the function `tree_reg_perf`, which takes in a DataFrame like `galton` and:
- Splits the data into training and test sets,
- Trains 20 decision trees – one for each depth between 1 and 20, and
- Computes both the training RMSE and test RMSE of each tree.

Return your results as a DataFrame that has two columns, `'train_err'` and `'test_err'`, and an index that corresponds to tree depths (i.e. 1, 2, ..., 20).

<br>

#### `knn_reg_perf` 🏘

A $k$-Nearest Neighbors ($k$-NN) regressor predicts the response value of a (new) observation by computing the **average response value** of the $k$ closest observations in the training set. The most common distance metric is Euclidean distance, i.e. $L_2$ distance, which we'll use here.

One **hyperparameter** of a $k$-NN regressor that affects model complexity is $k$, the **number of neighbors** averaged over. Larger values of $k$ correspond to more complicated regressors. We will explore this hyperparameter in this question.

Complete the implementation of the function `knn_reg_perf`, which takes in a DataFrame like `galton` and:
- Splits the data into training and test sets,
- Trains 20 $k$-NN regressors – one for each value of $k$ between 1 and 20, and
- Computes both the training RMSE and test RMSE of each regressor.

Again, return your results as a DataFrame that has two columns, `'train_err'` and `'test_err'`, and an index that corresponds to values of $k$ (i.e. 1, 2, ..., 20).

<br>

**Some guidelines for both subparts:**

- In all cases, we are using _all_ other columns in `galton` to predict `'childHeight'`.
- You need to import the necessary classes from `sklearn` **inside** the functions you create. Unlike before, we haven't imported them for you because we want you to figure out what to import! You'll need to do some outside research, such as reading the `sklearn` documentation, to answer this question.
- Use a test set size of 0.25. Within each function, you should only perform a single train-test split – that is, you should not be creating training and testing sets within a `for`-loop (though you will need to use a `for`-loop for something else).
- For this question, do not use any cross-validation.

In [53]:
# Use `galton` to test your work.
galton = pd.read_csv(Path('data') / 'galton.csv')
galton.head()

In [54]:
# don't change this cell, but do run it -- it is needed for the tests
galton_test = pd.read_csv(Path('data') / 'galton.csv')
out_tree_test = tree_reg_perf(galton_test)
out_knn_test = knn_reg_perf(galton_test)

In [ ]:
grader.check("q5")

After you've implemented both functions, run the cells below to plot training and testing error for both models.

In [66]:
pio.renderers.default = 'plotly_mimetype+notebook'

In [67]:
np.random.seed(9) # For reproducibility

tree = tree_reg_perf(galton)
knn = knn_reg_perf(galton)
hyp = np.arange(1, 21)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Error vs. Tree Depth for Decision Tree Regressor',
                                                    'Error vs. # Neighbors for k-NN Regressor'))

fig.add_trace(
    go.Scatter(x=hyp, y=tree.iloc[:, 0], name='Training Error'),
    row=1, col=1).add_trace(go.Scatter(x=hyp, y=tree.iloc[:, 1], name='Test Error'), 
                            row=1, col=1)

fig.add_trace(
    go.Scatter(x=hyp, y=knn.iloc[:, 0], line=dict(color='#1f77b4'), name='Training Error', showlegend=False),
    row=1, col=2).add_trace(go.Scatter(x=hyp, y=knn.iloc[:, 1], line=dict(color='#ff7f0f'),  name='Test Error',
                                       showlegend=False), row=1, col=2)

fig.update_layout(height=300, width=900)
fig.update_xaxes(title_text='Tree Depth', row=1, col=1, tickvals=np.arange(1,21,2))
fig.update_xaxes(title_text='# Neighbors', row=1, col=2, tickvals=np.arange(1,21,2))

fig.show()

If your training and evaluation routines are correct, you should notice a few things:
- In both models, test error initially decreases, and then (perhaps slowly) increases.
- With the decision tree, training error **decreases** as depth increases.
- With the $k$-NN regressor, training error **increases** as the number of neighbors increases.

You should think about **why** you observe each of the above phenomena. In particular, the last point may seem confusing – one might think that because larger values of $k$ correspond to more complicated models (because the regressor is looking at more information to make a prediction), larger values of $k$ should have lower training errors. But the nature of $k$-NN regressors is quite different than, say, decision tree regressors or linear regression models.

Lastly, in both cases, identify the ideal hyperparameter choice based on the graphs of test error. You don't have to submit this anywhere.

## Congratulations! You're done with Lab 9! 🏁

As a reminder, all of the work you want to submit needs to be in `lab.py`.

To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.

Once you've finished the lab, you should open the command line and run, in the directory for this lab:

```
python lab-validation.py
```

**This will run all of the `grader.check` cells that you see in this notebook, but only using the code in `lab.py` – that is, it doesn't look at any of the code in this notebook. If all of your `grader.check` cells pass in this notebook but not all of them pass in your command line with the above command, then you likely have code in your notebook that isn't in your `lab.py`!**

You can also use `lab-validation.py` to test individual questions. For instance,

```
python lab-validation.py q1 q2 q4
```

will run the `grader.check` cells for Questions 1, 2, and 4 – again, only using the code in `lab.py`. [This video](https://www.loom.com/share/0ea254b85b2745e59322b5e5a8692e91?sid=5acc92e6-0dfe-4555-9b6a-8115b6a52f99) how to use the script as well.

Once `python lab-validation.py` shows that you're passing all test cases, you're ready to submit your `lab.py` (and only your `lab.py`) to Gradescope. Once submitting to Gradescope, make sure to stick around until all test cases pass.

There is also a call to `grader.check_all()` below in _this_ notebook, but make sure to also follow the steps above.

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()